In [1]:
# NAME: Bruce Bai 
# GITHUB HANDLE: bruceb002 
# STUDENT ID: 1563103829 

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import calinski_harabasz_score
from sklearn.cluster import KMeans
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
# 1a) Download and split data into training and test sets
df = pd.read_csv("../data/Anuran Calls (MFCCs)/Frogs_MFCCs.csv")

features = df.iloc[:, :22] # 22 MFCCs
labels =  df.iloc[:, 22:25] # Family, Genus, Species
record_ids = df.iloc[:, -1:] # Record IDs column, if ever needed...

X_train, X_test, y_train, y_test = train_test_split(features, labels, train_size=0.7, random_state=1313)

In [3]:
## HELPER FUNCTIONS FOR THE REST OF THE HOMEWORK

# 1b)i. Researching exact match and hamming score / loss methods for evaluating multi-label classification
def evaluate_predictions(y_true, y_pred):
    correct = (y_true == y_pred)
    counts = correct.sum(axis=1)

    exact_match = (counts == 3).mean()
    hamming_loss = 1 - correct.sum().sum() / correct.size
    hamming_score = 1 - hamming_loss
    jaccard_score = (counts / (6 - counts)).mean()

    print(f"\nExact Match: {exact_match:.4f}")
    print(f"Hamming Loss: {hamming_loss:.4f}")
    print(f"Hamming Score (1 - Loss): {hamming_score:.4f}")
    print(f"Hamming Score (Jaccard): {jaccard_score:.4f}\n")
    
    return exact_match, hamming_loss, hamming_score, jaccard_score

print("    Exact match measures the proportion of instances where all labels (in our case: three labels - Family, Genus, and Species) are \
predicted correctly. However, it is a strict metric, as an instance is considered incorrect even if only one of its multiple labels is \
misclassified. Hamming loss addresses this limitation by measuring the proportion of individual labels that are predicted incorrectly, \
allowing for partial correctness. Hamming score is sometimes defined as '1 - Hamming loss', which measures the proportion of correctly \
predicted labels. Alternatively, Jaccard similarity is also used to evaluate multi-label classification by measuring the overlap between \
the actual and predicted label sets relative to their union. Since both approaches appear in different papers, we report both for \
comparison. Therefore, a higher exact match, Hamming score, or Jaccard similarity, and a lower Hamming loss, indicate better \
classification performance.\n")

# Re-usable helper function to modularize redundant coding logic
def train_and_evaluate(title, estimator, param_grid, c_key, X_train, X_test, y_train, y_test, cv, gamma_key=None):

    title_string = f"--- {title} ---".upper()
    print(title_string)
    print("-" * len(title_string))

    y_pred = pd.DataFrame(index=y_test.index)

    for label in y_train.columns:
        # Use label-specific grids if provided; otherwise use the common grid
        current_grid = param_grid[label] if label in param_grid else param_grid

        grid_search = GridSearchCV(estimator=estimator, param_grid=current_grid, cv=cv, scoring="accuracy", n_jobs=4)
        grid_search.fit(X_train, y_train[label])
        y_pred[label] = grid_search.predict(X_test)
        best_lambda = grid_search.best_params_[c_key]

        if gamma_key is not None:
            best_gamma = grid_search.best_params_[gamma_key]
            best_sigma = np.sqrt(1 / (2 * best_gamma))
            
            print(f"{label}: Best lambda = {best_lambda}, Best sigma = {best_sigma:.4f}, Best CV accuracy = {grid_search.best_score_:.4f}")
        else:
            print(f"{label}: Best lambda = {best_lambda}, Best CV accuracy = {grid_search.best_score_:.4f}")
            
    evaluate_predictions(y_test, y_pred)

    Exact match measures the proportion of instances where all labels (in our case: three labels - Family, Genus, and Species) are predicted correctly. However, it is a strict metric, as an instance is considered incorrect even if only one of its multiple labels is misclassified. Hamming loss addresses this limitation by measuring the proportion of individual labels that are predicted incorrectly, allowing for partial correctness. Hamming score is sometimes defined as '1 - Hamming loss', which measures the proportion of correctly predicted labels. Alternatively, Jaccard similarity is also used to evaluate multi-label classification by measuring the overlap between the actual and predicted label sets relative to their union. Since both approaches appear in different papers, we report both for comparison. Therefore, a higher exact match, Hamming score, or Jaccard similarity, and a lower Hamming loss, indicate better classification performance.



In [4]:
# 1b)ii. Training SVM with 10-fold CV Gaussian Kernel and OVA classifiers

# This 10-Fold CV variable will be re-used across the next couple of sub-problems
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=1313)

# estimator__C represents lambda values. The second list represent sigma values.
param_grids = {
    "Family": {"estimator__C": [1, 3, 5, 7, 10], "estimator__gamma": [1 / (2 * sigma**2) for sigma in [0.3, 0.35, 0.4]]},
    "Genus": {"estimator__C": [20, 30, 40, 50, 80], "estimator__gamma": [1 / (2 * sigma**2) for sigma in [0.55, 0.6, 0.65]]},
    "Species": {"estimator__C": [10, 20, 30, 40, 50], "estimator__gamma": [1 / (2 * sigma**2) for sigma in [0.45, 0.475, 0.5]]}
}

rbf_svm = OneVsRestClassifier(SVC(kernel="rbf"))

train_and_evaluate("Training SVM with 10-fold CV Gaussian Kernel and OVA classifiers", rbf_svm, param_grids, 
                    "estimator__C", X_train, X_test, y_train, y_test, cv, gamma_key="estimator__gamma")

--- TRAINING SVM WITH 10-FOLD CV GAUSSIAN KERNEL AND OVA CLASSIFIERS ---
------------------------------------------------------------------------


Family: Best lambda = 5, Best sigma = 0.3500, Best CV accuracy = 0.9944


Genus: Best lambda = 40, Best sigma = 0.6000, Best CV accuracy = 0.9919


Species: Best lambda = 30, Best sigma = 0.4750, Best CV accuracy = 0.9899

Exact Match: 0.9870
Hamming Loss: 0.0093
Hamming Score (1 - Loss): 0.9907
Hamming Score (Jaccard): 0.9895



In [5]:
# 1b)iii. Training via L1-penalized SVMs with 10-fold CV
l1_svm = Pipeline([("scaler", StandardScaler()), ("classifier", OneVsRestClassifier(LinearSVC(penalty="l1", dual=False, max_iter=20000)))])

lambda_values = [0.7, 0.715, 0.725, 1, 10]
param_grid_l1 = {"classifier__estimator__C": lambda_values}

train_and_evaluate("Training via L1-penalized SVMs with 10-fold CV", l1_svm, param_grid_l1, 
                   "classifier__estimator__C", X_train, X_test, y_train, y_test, cv)

--- TRAINING VIA L1-PENALIZED SVMS WITH 10-FOLD CV ---
------------------------------------------------------


Family: Best lambda = 0.7, Best CV accuracy = 0.9367


Genus: Best lambda = 10, Best CV accuracy = 0.9559


Species: Best lambda = 0.7, Best CV accuracy = 0.9595

Exact Match: 0.9157
Hamming Loss: 0.0517
Hamming Score (1 - Loss): 0.9483
Hamming Score (Jaccard): 0.9390



In [6]:
# 1b)iv. Training via SMOTE with 10-fold CV
l1_smote_svm = ImbPipeline([("scaler", StandardScaler()), ("smote", SMOTE(random_state=1313)),
    ("classifier", OneVsRestClassifier(LinearSVC(penalty="l1", dual=False, max_iter=100000)))])

lambda_values = [0.6, 0.7, 1, 5, 10]
param_grid_smote = {"classifier__estimator__C": lambda_values}

train_and_evaluate("Training via SMOTE with 10-fold CV", l1_smote_svm, param_grid_smote, 
                   "classifier__estimator__C", X_train, X_test, y_train, y_test, cv)

--- TRAINING VIA SMOTE WITH 10-FOLD CV ---
------------------------------------------


Family: Best lambda = 0.6, Best CV accuracy = 0.9188


Genus: Best lambda = 5, Best CV accuracy = 0.9152


Species: Best lambda = 0.6, Best CV accuracy = 0.9597

Exact Match: 0.8541
Hamming Loss: 0.0746
Hamming Score (1 - Loss): 0.9254
Hamming Score (Jaccard): 0.9061



In [1]:
print("    After applying SMOTE to the L1-penalized SVMs, the exact-match score decreased from 0.9157 to 0.8541, while the Hamming loss \
increased from 0.0517 to 0.0746. The cross-validation accuracies for Family and Genus also decreased, while Species remained \
approximately the same. This suggests that SMOTE did not improve the overall classification performance in our experiment. One \
possible explanation is that balancing the classes caused the classifiers to favor minority classes more, potentially at the expense \
of correctly predicting the majority classes. However, there would need to be more performance analyzation of individual classes to \
confirm this.\n")

print("    Comparing all three approaches, the Gaussian Kernel SVMs performed the best, achieving an exact-match score of 0.9870 and a \
Hamming loss of 0.0093. The original L1-penalized SVMs performed second best, followed by the L1-penalized SVMs with SMOTE. This suggests \
that the Gaussian Kernel SVMs were better suited for this classification task, possibly because they can capture nonlinear relationships \
between the MFCC features. Overall, while SMOTE was intended to address class imbalance, it did not improve the overall performance of \
the L1-penalized SVMs based on our evaluation metrics.")

    After applying SMOTE to the L1-penalized SVMs, the exact-match score decreased from 0.9157 to 0.8541, while the Hamming loss increased from 0.0517 to 0.0746. The cross-validation accuracies for Family and Genus also decreased, while Species remained approximately the same. This suggests that SMOTE did not improve the overall classification performance in our experiment. One possible explanation is that balancing the classes caused the classifiers to favor minority classes more, potentially at the expense of correctly predicting the majority classes. However, there would need to be more performance analyzation of individual classes to confirm this.

    Comparing all three approaches, the Gaussian Kernel SVMs performed the best, achieving an exact-match score of 0.9870 and a Hamming loss of 0.0093. The original L1-penalized SVMs performed second best, followed by the L1-penalized SVMs with SMOTE. This suggests that the Gaussian Kernel SVMs were better suited for this classification 

In [8]:
# 2) Monte Carlo 50 Simulations: K-means clustering
results = []
    
for i in range(50):
    best_score = -float('inf')

    # 2a) Choose k = {2, 3, ... , 50} automatically using CH
    # CH is not defined with only one cluster, so we begin at k=2
    for k in range(2, 51):
        seed = 1313+i
        kmeans = KMeans(n_clusters=k, n_init=1, random_state=seed)
        clusters = kmeans.fit_predict(features)
        CH_score = calinski_harabasz_score(features, clusters)

        if CH_score > best_score:
            best_score = CH_score
            best_k = k
            best_clusters = clusters

    # 2b) Determine the majority Family, Genus, and Species per cluster
    cluster_groups = labels.groupby(best_clusters)
    majority_labels = pd.DataFrame(columns=labels.columns)
    
    for cluster, cluster_data in cluster_groups:
        majority_labels.loc[cluster] = cluster_data.mode().iloc[0]

    if i == 0:
        print("Majority labels for each cluster:")
        print(majority_labels)
    
    y_pred = majority_labels.loc[best_clusters]
    y_pred = y_pred.reset_index(drop=True)

    # 2c) Calculate Hamming distance, score, loss, and Jaccard similarity
    correct = (labels == y_pred)
    counts = correct.sum(axis=1)
    
    hamming_distance = (3 - counts).mean()
    hamming_loss = 1 - correct.to_numpy().mean()
    hamming_score = 1 - hamming_loss
    jaccard_score = (counts / (6 - counts)).mean()
    
    results.append({
        "k": best_k,
        "Hamming Distance": hamming_distance,
        "Hamming Loss": hamming_loss,
        "Hamming Score (1 - Loss)": hamming_score,
        "Hamming Score (Jaccard)": jaccard_score
    })

results_df = pd.DataFrame(results)

title = "--- K-MEANS MONTE CARLO RESULTS (50 RUNS) ---"
print(title)
print("-" * len(title))

for metric in results_df.columns:
    avg = results_df[metric].mean()
    std_dev = results_df[metric].std()
    print(f"{metric}: Average = {avg:.4f}, Standard Deviation = {std_dev:.4f}")
    
print("")
print("Using the Calinski–Harabasz (CH) criterion to select the optimal number of clusters, K-means consistently selected k = 2 across \
all 50 Monte Carlo simulations. The average Hamming distance was 0.8956, indicating approximately 0.90 incorrectly assigned labels per \
observation out of the three labels (Family, Genus, and Species). The average Hamming loss was 0.2985, while the Hamming score (1 − loss) \
and Jaccard similarity were 0.7015 and 0.6819, respectively.\n")

print("Interestingly, all standard deviations were 0, meaning that all 50 simulations produced identical evaluation metrics despite using \
different random seeds. Further inspection showed that the CH criterion consistently favored k = 2, and comparing two different \
initializations yielded an Adjusted Rand Index of 1.0000, indicating identical cluster assignments for those two runs. The majority-label \
distributions were also consistent across the simulations we inspected, suggesting that the clustering was highly stable. However, \
since only two clusters were selected despite the dataset containing multiple families, genera, and species, the assigned majority \
labels could not fully represent all the different classes, which may explain the relatively high Hamming loss.")

Majority labels for each cluster:
            Family      Genus                 Species
0  Leptodactylidae  Adenomera  AdenomeraHylaedactylus
1          Hylidae  Hypsiboas       HypsiboasCordobae


--- K-MEANS MONTE CARLO RESULTS (50 RUNS) ---
---------------------------------------------
k: Average = 2.0000, Standard Deviation = 0.0000
Hamming Distance: Average = 0.8956, Standard Deviation = 0.0000
Hamming Loss: Average = 0.2985, Standard Deviation = 0.0000
Hamming Score (1 - Loss): Average = 0.7015, Standard Deviation = 0.0000
Hamming Score (Jaccard): Average = 0.6819, Standard Deviation = 0.0000

Using the Calinski–Harabasz (CH) criterion to select the optimal number of clusters, K-means consistently selected k = 2 across all 50 Monte Carlo simulations. The average Hamming distance was 0.8956, indicating approximately 0.90 incorrectly assigned labels per observation out of the three labels (Family, Genus, and Species). The average Hamming loss was 0.2985, while the Hamming score (1 − loss) and Jaccard similarity were 0.7015 and 0.6819, respectively.

Interestingly, all standard deviations were 0, meaning that all 50 simulations produced identical evaluation metrics despite usi

### AI Attribution Statement

**Works Cited**

OpenAI. *ChatGPT*. Responses to prompts regarding Python debugging, code modularization, SVM hyperparameter tuning, evaluation metrics, and clustering concepts, November 2026, https://chatgpt.com/.

**Usage Summary & Contributions**

In accordance with course policy, Generative AI (ChatGPT) was utilized as an educational assistance tool to support programming comprehension, debugging, and interpretation of experimental results. Specifically, assistance included:

* **Debugging & Modularization:** Reviewing code for logical correctness, verifying consistency with library documentation, and exploring reusable helper functions to reduce redundant code.
* **Hyperparameter Strategy:** Discussing appropriate search ranges for SVM penalty ($\lambda$) and Gaussian kernel width ($\sigma$), and interpreting cross-validation results to understand parameter sensitivity.
* **Concept Clarification:** Differentiating between Hamming score ($1 - \text{loss}$) and Jaccard similarity, as well as clarifying concepts related to SMOTE, K-means clustering, and hierarchical linkage.
* **Write-up Articulation:** Assisting with organizing and refining analytical explanations based on observed experimental results and independently verified calculations.

### Representative AI Prompts Used

The following are representative prompts and excerpts from AI-assisted discussions throughout the assignment:

1. **SVM Hyperparameter Tuning:** "Thoughts on this new list: `lambda_values = [0.6, 0.65, 0.7, 0.75, 0.8, 1, 5, 10, 15]`?"

2. **Convergence Debugging:** "How should we resolve the convergence warning? How come there was a warning?"

3. **Code Modularization:** "Is it possible to write one helper function that makes all the redundant code not redundant, with all the necessary input arguments so that we can re-use the helper function across all parts ii, iii, and iv?"

4. **Monte Carlo Debugging:** "Is it possible we're gathering or printing out stats in the wrong place with how we implemented it?" (Accompanied by the K-means implementation.)

5. **Clustering Analysis:** "Based on all the analytical debugging we've done, why do you think all the standard deviations were 0?"

*The final implementation was reviewed, adapted, executed, and validated by the student, who also critically evaluated the experimental findings.*